# Frequency-Stratified Integrated Gradients

This notebook implements a frequency-aware explainability method by running Integrated Gradients (IG)
with **blurred baselines at increasing σ levels**. By differencing attributions between consecutive
blur levels, we isolate *which spatial frequency bands* the model relied on to make its prediction.

**Pipeline:**
1. Load a pretrained model + image
2. Generate baselines: blurred versions of the image at σ = [0, 2, 4, 8, 16, 32]
3. For each σ, compute IG attribution (integral of gradients from blurred → original)
4. Differential attributions: `ΔAttr(band_i) = Attr(σᵢ) - Attr(σᵢ₊₁)` → captures contribution of the removed frequency band
5. Visualise all bands + aggregate in a single inferno heatmap

In [ ]:
# ── Dependencies ─────────────────────────────────────────────────────────────
# pip install torch torchvision Pillow matplotlib scipy numpy

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize

import torch
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models

from PIL import Image
from scipy.ndimage import gaussian_filter

print('All imports OK')

## 1 — Configuration

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
import os

DATA_DIR = '/home/arin_weling/wavex/wavelet_explanation/data'
OUTPUT_DIR = '/home/arin_weling/wavex/notebooks/outputs/freq_ig_explanation'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Images to process
IMAGE_NAMES = ['husky_lion','shih_tzu', 'husky_cat', 'cobra', 'banded_gecko', 'peacock', 'rock_python']

# ── Baseline mode ──────────────────────────────────────────────────────────────
# 'gaussian' : blur the image with Gaussian at increasing σ levels (pixel space)
# 'fourier'  : low-pass filter via hard square mask in Fourier space
BASELINE_MODE = 'fourier'   # 'gaussian' | 'fourier'

# ── Gaussian mode: sigma levels ────────────────────────────────────────────────
# Each consecutive pair (σᵢ, σᵢ₊₁) defines one band.
# σ=0  → original (no blur)  → highest frequencies present
# σ=32 → heavily blurred     → only very low frequencies remain
SIGMA_LEVELS = [0, 1, 2, 3, 4, 5]

# ── Fourier mode: band configuration ──────────────────────────────────────────
# Number of frequency bands to split the spectrum into.
N_BANDS = 5

# Cutoff radius strategy (only used when BASELINE_MODE = 'fourier'):
#   'adaptive'   → equal cumulative FFT-energy bands (image-specific)
#   'geometric'  → log-spaced radii between GEOM_MIN_CUTOFF and max_r
#   'octave'     → powers-of-two spacing (1, 2, 4, 8, ...)
CUTOFF_MODE     = 'geometric'  # 'adaptive' | 'geometric' | 'octave'
GEOM_MIN_CUTOFF = 5            # smallest cutoff radius for geometric spacing

IG_STEPS     = 100    # number of interpolation steps for IG (higher = more accurate)
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
print(f'Images to process: {IMAGE_NAMES}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Baseline mode: {BASELINE_MODE}')
if BASELINE_MODE == 'gaussian':
    print(f'Sigma levels: {SIGMA_LEVELS}')
else:
    print(f'Cutoff mode: {CUTOFF_MODE}  |  N_BANDS={N_BANDS}  |  GEOM_MIN_CUTOFF={GEOM_MIN_CUTOFF}')

## 2 — Load Model

In [ ]:
def load_model(device):
    """Load a pretrained ResNet-18. Swap for any torchvision model."""
    # model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.eval()
    model.to(device)
    return model

model = load_model(DEVICE)
print('Model loaded — ResNet-18 (ImageNet)')

## 3 — Image Loading & Preprocessing

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

def load_image(path):
    """Load image and return (PIL image, normalised tensor [1,3,H,W])."""
    pil_img = Image.open(path).convert('RGB')
    pil_img = pil_img.resize((224, 224), Image.BILINEAR)
    tensor  = preprocess(pil_img).unsqueeze(0)
    return pil_img, tensor

print('Preprocessing pipeline defined.')

## 4 — Determine Target Class

In [ ]:
# ── Set this to explore a single image ────────────────────────────────────────
IMAGE_PATH = os.path.join(DATA_DIR, 'husky_cat.jpg')   # <-- change as needed
TARGET_CLASS = None  # None = use top-1 prediction

# ── Load & display ─────────────────────────────────────────────────────────────
pil_img, img_tensor = load_image(IMAGE_PATH)
img_tensor = img_tensor.to(DEVICE)
print(f'Image tensor shape: {img_tensor.shape}')

plt.figure(figsize=(4, 4))
plt.imshow(pil_img)
plt.axis('off')
plt.title('Input image')
plt.tight_layout()
plt.show()

# ── Determine target class ─────────────────────────────────────────────────────
with torch.no_grad():
    logits = model(img_tensor)
    probs  = F.softmax(logits, dim=1)

top5_prob, top5_idx = probs.topk(5, dim=1)
print('Top-5 predictions:')
for i, (p, idx) in enumerate(zip(top5_prob[0], top5_idx[0])):
    print(f'  {i+1}. class {idx.item():>4d}  ({p.item()*100:.2f}%)')

if TARGET_CLASS is None:
    TARGET_CLASS = top5_idx[0, 0].item()
print(f'\nUsing target class: {TARGET_CLASS}')

## 5 — Blurred Baselines

For each σ we apply a Gaussian blur **in pixel space** (before normalisation) and then renormalise.
σ=0 means no blur → the original image acts as the "sharpest" baseline.

In [ ]:
# Cache normalization tensors per device to avoid repeated allocation
_NORM_CACHE = {}

def _get_norm_tensors(device):
    """Get cached mean/std tensors for the given device."""
    if device not in _NORM_CACHE:
        _NORM_CACHE[device] = (
            torch.tensor(IMAGENET_MEAN, device=device, dtype=torch.float32).view(1, 3, 1, 1),
            torch.tensor(IMAGENET_STD, device=device, dtype=torch.float32).view(1, 3, 1, 1)
        )
    return _NORM_CACHE[device]


def load_image(path):
    """Load image and return (PIL image, normalised tensor [1,3,H,W])."""
    pil_img  = Image.open(path).convert('RGB')
    pil_img  = pil_img.resize((224, 224), Image.BILINEAR)
    tensor   = preprocess(pil_img).unsqueeze(0)
    return pil_img, tensor


# ── Gaussian baseline ──────────────────────────────────────────────────────────

def blur_tensor(tensor, sigma):
    """
    Apply Gaussian blur to a normalised image tensor [1,3,H,W].
    We un-normalise → blur in pixel space → re-normalise.
    Blur is done on GPU using torchvision to avoid CPU round-trips.
    sigma=0 returns the tensor unchanged.
    """
    if sigma == 0:
        return tensor.clone()

    mean, std = _get_norm_tensors(tensor.device)

    # Un-normalise to [0, 1]
    pixel = tensor * std + mean  # (1, 3, H, W)

    # GPU-native Gaussian blur to avoid CPU transfers
    kernel_size = int(2 * np.ceil(3 * sigma) + 1)  # std rule: ±3σ
    kernel_size = kernel_size if kernel_size % 2 == 1 else kernel_size + 1
    from torchvision.transforms.functional import gaussian_blur
    blurred = gaussian_blur(pixel, kernel_size=(kernel_size, kernel_size), sigma=sigma)

    # Re-normalise
    blurred = (blurred - mean) / std
    return blurred


# ── Fourier baseline ───────────────────────────────────────────────────────────

def lowpass_tensor(tensor, cutoff):
    """
    Square low-pass filter a normalised image tensor [1,3,H,W] via FFT.

    Steps:
      1. Un-normalise to pixel space [0, 1]
      2. FFT2 each channel (DC shifted to centre)
      3. Zero out all coefficients outside a square of side 2*cutoff+1 centred on DC
      4. IFFT2 back to pixel space, take real part
      5. Re-normalise with ImageNet stats

    cutoff=0      → DC only → uniform mean-colour image (maximally blurred)
    cutoff>=H//2  → full spectrum → original image
    """
    H, W = tensor.shape[-2], tensor.shape[-1]
    if cutoff >= H // 2:
        return tensor.clone()

    mean, std = _get_norm_tensors(tensor.device)

    # Un-normalise to [0, 1]
    pixel = (tensor * std + mean).squeeze(0)   # (3, H, W)

    # FFT with DC shifted to centre
    fft_shifted = torch.fft.fftshift(
        torch.fft.fft2(pixel), dim=(-2, -1)
    )  # (3, H, W) complex

    # Build square mask centred at (cy, cx)
    cy, cx = H // 2, W // 2
    mask = torch.zeros(H, W, dtype=torch.bool, device=tensor.device)
    mask[max(0, cy - cutoff): cy + cutoff + 1,
         max(0, cx - cutoff): cx + cutoff + 1] = True

    # Zero out everything outside the square
    fft_filtered = fft_shifted * mask.unsqueeze(0)  # (3, H, W)

    # IFFT back to pixel space
    pixel_filtered = torch.fft.ifft2(
        torch.fft.ifftshift(fft_filtered, dim=(-2, -1))
    ).real.clamp(0, 1)  # (3, H, W)

    # Re-normalise and restore batch dim
    return ((pixel_filtered.unsqueeze(0) - mean) / std)


def compute_cutoffs(tensor, n_bands, mode='geometric', geometric_min_r=1):
    """
    Compute n_bands+1 cutoff radii that divide the Fourier spectrum into n_bands bands.

    Parameters
    ----------
    tensor : torch.Tensor [1,3,H,W]  (used only for adaptive energy computation)
    n_bands : int
    mode : 'adaptive' | 'geometric' | 'octave'
    geometric_min_r : int  smallest radius for geometric spacing

    Returns
    -------
    list of int, length n_bands+1
        [r_0, r_1, ..., r_n]  where r_0 is the smallest cutoff and
        r_n >= H//2 (full spectrum = original image).
    """
    H = tensor.shape[-2]
    max_r = H // 2

    if mode == 'geometric':
        # Log-spaced radii from geometric_min_r to max_r
        radii = np.geomspace(geometric_min_r, max_r, n_bands + 1)
        radii = sorted(set(int(round(r)) for r in radii))
        if radii[-1] < max_r:
            radii[-1] = max_r

    elif mode == 'octave':
        # Powers-of-two spacing: 1, 2, 4, 8, ...
        radii = []
        r = 1
        while r < max_r:
            radii.append(r)
            r *= 2
        radii.append(max_r)
        if len(radii) > n_bands + 1:
            radii = radii[:n_bands] + [max_r]
        while len(radii) < n_bands + 1:
            radii.insert(-1, (radii[-2] + radii[-1]) // 2)

    elif mode == 'adaptive':
        # Equal cumulative FFT energy per band (image-specific)
        mean, std = _get_norm_tensors(tensor.device)
        pixel = (tensor * std + mean).squeeze(0)  # (3, H, W)
        fft_mag = torch.fft.fftshift(
            torch.abs(torch.fft.fft2(pixel)), dim=(-2, -1)
        ).mean(0)  # (H, W) average across channels

        cy, cx = H // 2, tensor.shape[-1] // 2
        energy_at_r = []
        for r in range(max_r + 1):
            lo_r, hi_r = max(0, cy - r), cy + r + 1
            lo_c, hi_c = max(0, cx - r), cx + r + 1
            energy_at_r.append(fft_mag[lo_r:hi_r, lo_c:hi_c].sum().item())

        total = energy_at_r[-1]
        targets = [total * (i + 1) / n_bands for i in range(n_bands)]
        radii = [1]
        for target in targets:
            r = next((i for i, e in enumerate(energy_at_r) if e >= target), max_r)
            radii.append(r)
        radii[-1] = max_r
        radii = sorted(set(radii))
        if len(radii) > n_bands + 1:
            radii = radii[:n_bands] + [max_r]

    else:
        raise ValueError(f"Unknown CUTOFF_MODE: {mode!r}. Choose 'adaptive', 'geometric', or 'octave'.")

    return [int(r) for r in radii]


print('Blur, Fourier low-pass, and compute_cutoffs functions defined.')

## 6 — Integrated Gradients

For a given baseline `x'` we compute:

$$\text{IG}_i(x) = (x_i - x'_i) \times \int_{\alpha=0}^{1} \frac{\partial F(x' + \alpha(x - x'))}{\partial x_i} \, d\alpha$$

Approximated via the **Riemann sum** over `IG_STEPS` steps.

In [ ]:
def integrated_gradients(model, img, baseline, target_class, steps=50, batch_size=10):
    """
    Compute IG attributions with proper Riemann approximation and memory-efficient batching.

    Formula:
        IG_i(x) = (x_i - x'_i) × ∫₀¹ ∇_i F(x' + α(x - x')) dα
    
    Approximated via Riemann sum: IG ≈ (x - x') × (1/steps) × Σ ∇F(interp_α)

    Parameters
    ----------
    batch_size : int, default=10
        Process this many interpolation steps per forward/backward pass.
        Reduces to avoid GPU memory overflow on large models. Smaller = safer but slower.

    Returns
    -------
    attribution : np.ndarray, shape (H, W)
        Absolute attribution magnitude (summed over channels).
    attribution_raw : np.ndarray, shape (3, H, W)
        Signed attribution per channel.
    """
    # Detach inputs to ensure we only backprop through interpolation
    baseline = baseline.detach()
    img = img.detach()

    # Build interpolation schedule and delta
    alphas = torch.linspace(0, 1, steps + 1, device=img.device)
    delta = img - baseline  # (1, 3, H, W)

    # Accumulate gradients across batches
    grads_all = []

    for i in range(0, len(alphas), batch_size):
        alpha_batch = alphas[i : i + batch_size]
        
        # Interpolate: baseline + α·Δ
        interp = baseline + alpha_batch.view(-1, 1, 1, 1) * delta  # (batch_size, 3, H, W)
        interp.requires_grad_(True)

        # Forward pass
        logits = model(interp)  # (batch_size, num_classes)
        score = logits[:, target_class].sum()

        # Backward
        score.backward()

        # Collect batch gradients
        grads_all.append(interp.grad.detach())

    # Concatenate all batch gradients
    grads = torch.cat(grads_all, dim=0)  # (steps+1, 3, H, W)

    # Riemann sum: ∫ ∇ dα ≈ (1/steps) × Σ ∇_i  →  avg across all interpolation points
    avg_grads = grads.mean(dim=0, keepdim=True)  # (1, 3, H, W)

    # Attribution: (x - baseline) × avg_grad
    ig_raw = (delta * avg_grads).squeeze(0).cpu().numpy()  # (3, H, W)
    ig_abs = np.abs(ig_raw).sum(axis=0)  # (H, W)

    return ig_abs, ig_raw


def normalise(arr, clip_percentile=99):
    """Normalize array to [0, 1] by clipping high outliers and scaling."""
    arr = np.maximum(arr, 0)
    vmax = np.percentile(arr, clip_percentile)
    if vmax == 0:
        return arr
    return arr / vmax


print('All functions defined.')

## 7 — Compute IG for Every Sigma Level

In [ ]:
# ════════════════════════════════════════════════════════════════════════════════
# MAIN LOOP: Process all images
# ════════════════════════════════════════════════════════════════════════════════

for image_name in IMAGE_NAMES:
    print(f"\n{'='*80}")
    print(f"Processing: {image_name}")
    print(f"{'='*80}")

    # ── Find image file ───────────────────────────────────────────────────────
    image_path = None
    for ext in ['.jpg', '.png', '.jpeg', '.webp']:
        candidate = os.path.join(DATA_DIR, image_name + ext)
        if os.path.exists(candidate):
            image_path = candidate
            break

    if image_path is None:
        print(f"❌ Image not found: {image_name}")
        continue

    print(f"✓ Found: {image_path}")

    # ── Load image ────────────────────────────────────────────────────────────
    pil_img, img_tensor = load_image(image_path)
    img_tensor = img_tensor.to(DEVICE)
    print(f"✓ Loaded image tensor shape: {img_tensor.shape}")

    # ── Determine target class ────────────────────────────────────────────────
    with torch.no_grad():
        logits = model(img_tensor)
        probs  = F.softmax(logits, dim=1)

    top5_prob, top5_idx = probs.topk(5, dim=1)
    target_class = top5_idx[0, 0].item()
    top1_prob    = top5_prob[0, 0].item()
    print(f"✓ Top-1 class: {target_class} ({top1_prob*100:.2f}%)")

    # ── Generate baselines (mode-dependent) ───────────────────────────────────
    if BASELINE_MODE == 'gaussian':
        levels    = SIGMA_LEVELS
        baselines = [blur_tensor(img_tensor, s) for s in levels]
        band_label_fn = lambda i: f'σ {levels[i]}→{levels[i+1]}'
        mode_str  = 'Gaussian'
        level_desc = f'σ levels: {levels}'
    else:  # fourier
        levels    = compute_cutoffs(img_tensor, n_bands=N_BANDS, mode=CUTOFF_MODE,
                                    geometric_min_r=GEOM_MIN_CUTOFF)
        baselines = [lowpass_tensor(img_tensor, r) for r in levels]
        band_label_fn = lambda i: f'r {levels[i]}→{levels[i+1]}'
        mode_str  = f'Fourier ({CUTOFF_MODE})'
        level_desc = f'cutoff radii: {levels}'

    print(f"✓ Baseline mode: {mode_str}  |  {level_desc}")
    print(f"✓ Created {len(baselines)} baselines")

    # ── Compute IG for each level ─────────────────────────────────────────────
    ig_results = []
    for i, (level, baseline) in enumerate(zip(levels, baselines)):
        label = f'σ={level}' if BASELINE_MODE == 'gaussian' else f'r={level}'
        print(f"  [{i+1}/{len(levels)}] Computing IG for {label}...", end=' ')
        attr_abs, attr_raw = integrated_gradients(
            model, img_tensor, baseline, target_class, steps=IG_STEPS
        )
        ig_results.append(attr_abs)
        print(f"max={attr_abs.max():.4f}")

    # ── Compute differential band attributions ────────────────────────────────
    band_attributions = []
    band_labels       = []

    for i in range(len(levels) - 1):
        diff = ig_results[i] - ig_results[i + 1]
        band_attributions.append(diff)
        band_labels.append(band_label_fn(i))

    n_bands = len(band_attributions)
    print(f"✓ Computed {n_bands} frequency bands")

    # ── Visualize: Frequency Band Attributions ────────────────────────────────
    clamped_bands    = [np.maximum(b, 0) for b in band_attributions]
    band_stack       = np.stack(clamped_bands, axis=0)
    dominant_band    = np.argmax(band_stack, axis=0)
    max_contribution = np.max(band_stack, axis=0)

    total_panels = 1 + n_bands + 1
    fig_w        = 3.2 * total_panels
    fig, axes    = plt.subplots(1, total_panels, figsize=(fig_w, 4),
                                facecolor='#0d0d0d')

    CMAP      = 'inferno'
    all_vals  = np.concatenate([np.maximum(b, 0).ravel() for b in band_attributions])
    vmax_glob = np.percentile(all_vals, 99)

    # Panel 0: original image
    mean_np  = np.array(IMAGENET_MEAN).reshape(3, 1, 1)
    std_np   = np.array(IMAGENET_STD ).reshape(3, 1, 1)
    vis_orig = (img_tensor.squeeze(0).cpu().numpy() * std_np + mean_np).clip(0, 1).transpose(1, 2, 0)

    axes[0].imshow(vis_orig)
    axes[0].set_title('Original', color='white', fontsize=10, pad=6)
    axes[0].axis('off')

    # Panels 1..n_bands: per-band attributions
    for i, (band, label) in enumerate(zip(band_attributions, band_labels)):
        ax   = axes[i + 1]
        data = np.maximum(band, 0)
        im   = ax.imshow(data, cmap=CMAP, vmin=0, vmax=vmax_glob)
        ax.set_title(label, color='white', fontsize=9, pad=6)
        ax.axis('off')
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.yaxis.set_tick_params(color='white', labelcolor='white', labelsize=6)
        cbar.outline.set_edgecolor('white')

    # Last panel: dominant band
    ax_dom        = axes[-1]
    cmap_spectrum = matplotlib.colormaps.get_cmap('hsv')
    colors        = [cmap_spectrum(1.0 - (i + 1) / n_bands) for i in range(n_bands)]

    dom_rgb = np.zeros((dominant_band.shape[0], dominant_band.shape[1], 3))
    for band_idx in range(n_bands):
        mask = dominant_band == band_idx
        dom_rgb[mask] = colors[band_idx][:3]

    contribution_norm = normalise(max_contribution)
    dom_rgb_bright    = np.clip(dom_rgb * contribution_norm[..., None], 0, 1)

    ax_dom.imshow(dom_rgb_bright)
    ax_dom.set_title('Dominant Band', color='white', fontsize=10, pad=6, fontweight='bold')
    ax_dom.axis('off')

    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=colors[i][:3], label=band_labels[i])
                       for i in range(n_bands)]
    ax_dom.legend(handles=legend_elements, loc='upper left', fontsize=7,
                  bbox_to_anchor=(0, 1), framealpha=0.9, facecolor='#0d0d0d',
                  edgecolor='white', frameon=True)

    fig.suptitle(
        f'Frequency-Stratified IG [{mode_str}]  |  {image_name}  |  class {target_class}',
        color='white', fontsize=12, y=1.01
    )

    plt.tight_layout()

    # ── Save with image name ──────────────────────────────────────────────────
    output_filename = os.path.join(OUTPUT_DIR, f'freq_ig_explanation_{image_name}.png')
    plt.savefig(output_filename, dpi=150, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    print(f"✓ Saved: {output_filename}")
    plt.close()

print(f"\n{'='*80}")
print(f"✓ All {len(IMAGE_NAMES)} images processed and saved to {OUTPUT_DIR}")
print(f"{'='*80}")

## 8 — Differential Frequency Band Attributions

The attribution map at σᵢ captures everything the model uses that is *above* the σᵢ cutoff frequency.

So the **differential**:

$$\Delta\text{Attr}(\text{band}_i) = \text{Attr}(\sigma_i) - \text{Attr}(\sigma_{i+1})$$

isolates the contribution of the frequency content that was present at σᵢ but absent at σᵢ₊₁.
Lower σ bands = high-frequency details; higher σ bands = coarse/low-frequency structure.

## 9 — Visualise: Frequency Band Attributions + Aggregate

Each panel shows **where in the image** the model used information in that frequency band.

The last panel is the **weighted aggregate** — a single importance map across all frequencies.

## 10 — Frequency Importance Bar Chart

A global scalar importance score per band — answers *'which frequency range mattered most overall?'*

---
## Interpretation Guide

| Band σ range | Frequency content | What a high score means |
|---|---|---|
| Low σ (e.g. 0→2) | Very high frequencies — fine edges, grain | Model is **texture-biased** |
| Mid σ (e.g. 5→10) | Mid frequencies — object parts, contours | Model uses **shape features** |
| High σ (e.g. 20→40) | Low frequencies — colour blobs, silhouette | Model relies on **global structure** |

**Tips:**
- If most attribution is in low-σ bands → model is texture-biased (common in standard ImageNet CNNs)
- If attribution spreads into high-σ bands → model is shape/structure-biased (common after Stylized-ImageNet training)
- Bright hotspots in a high-σ panel but not low-σ = the region matters for its **coarse shape**, not fine detail
- You can swap `SIGMA_LEVELS` for finer or coarser frequency analysis